# 22 — O1-S exact aggregate + LB64 comparison (GPU, DEV-only)

Notebook này chạy **một lần** sau khi cả notebook 21 của Quang và Trọng đều có
manifest `COMPLETE`. Nó xác minh lại 35 marker, toàn bộ hash artifact và đúng
32.867 exact frame; sau đó đo best source trên đủ từng frame của 61 interval dài.

Mười DEV event có interval không quá 64 frame đã được notebook 20 quét đầy đủ;
notebook 22 giữ nguyên kết quả đã khóa đó và thay 61 lower-bound event bằng exact
result. Đầu ra cuối có đủ 71 DEV query-event, so exact với LB64, đo score/rank và
phân loại failure ở biên SigLIP dense.

Đây vẫn là `diagnostic_only`: không đọc held-out, không ghi `keyframes-btc`, không
ingest live index và không được gọi là official-score uplift. OCR/caption/BGE,
fusion, reranker và scorer vẫn phải qua parity gate trước notebook 23.

### Cách chạy

1. Import notebook từ `feature/quang/storage`, chọn GPU T4/L4/A10 và bật Internet.
2. Bật Secrets `GITHUB_TOKEN`, `MODAL_TOKEN_ID`, `MODAL_TOKEN_SECRET`.
3. Attach đúng Dataset `mfe-data-v2` như notebook 20 nếu BTC flat SigLIP chưa có
   trên Modal; expanded layout hoặc `mfe-data-v2.zip` đều được.
4. Giữ nguyên `RUN_SCOPE="AGGREGATE"`, bấm **Save Version / Run All** một lần.
5. Thành công khi log cuối có `O1S_EXACT_AGGREGATE_COMPLETE`, `events=71`,
   `exact_events=61` và manifest remote đã verify.

## 0 — Khóa aggregate; không chia runner

In [ ]:
from pathlib import Path

RUN_SCOPE = "AGGREGATE"
CONFIRM_AGGREGATE = "O1S-EXACT-DEV"
RUN_ID = "o1s-exact-aggregate-v1"
SCAN_RUN_ID = "o1s-exact-v1"
RUNNERS = ("quang", "trong")
EXACT_POLICY = "inclusive_full_interval_exact"
LB64_POLICY = "inclusive_uniform_integer_lb64"
MAX_SAMPLES_PER_EVENT = 64
EXPOSURE_DEPTH = 100
SIGNAL_IMPROVEMENT_EPSILON = 1e-6

EXPECTED_EXACT_PLAN_HASH = "1c1318d5c40d71176176eb0d4ee5d1306159f0e898110118d6e27099e086a35c"
EXPECTED_EXACT_REPO_COMMIT = "6dc5865a046a3329323cb8606ec93e7802173e51"
EXPECTED_NB18_REPORT_SHA256 = "b089835a679809cfb1d2a1f79580343a8ceb0fe34846452038101f0db3a59228"
EXPECTED_TARGETS_SHA256 = "6929818a00205179f1d96c394f2930f43f88e9d536edccaf169b0632b5ee716b"
EXPECTED_NB20_COMPLETE_SHA256 = "27349b5a3b2aa0e2858934106808224bef893cf938e01616c0bb8d9e647b4804"
EXPECTED_SIGLIP_CANARY_REPORT_HASH = "45828774efcb0e291538b3405e41c8c720c9bb4e5c4e6588401c7a0547ca5a3c"
EXPECTED_TARGETS = 219
EXPECTED_DEV_QUERIES = 63
EXPECTED_DEV_TARGETS = 71
EXPECTED_EXACT_TARGETS = 61
EXPECTED_EXACT_VIDEOS = 35
EXPECTED_EXACT_FRAMES = 32867
EXPECTED_BTC_SIGLIP_ROWS = 176707

CHECKPOINT = "google/siglip-so400m-patch14-384"
MODEL_REVISION = "9fdffc58afc957d1a03a25b10dba0329ab15c2a3"
SIGLIP_DIMENSION = 1152
TEXT_MAX_LENGTH = 64
TEXT_BATCH_SIZE = 32

REPO_URL = "https://github.com/AIVIETNAM-AIO-Triet-Descartes/multimodal-fusion-engine.git"
REPO_REF = "feature/quang/storage"
REPO = Path("/kaggle/working/multimodal-fusion-engine")
WORK = Path("/kaggle/working/o1s-exact-aggregate")
REMOTE = WORK / "remote"
OUTPUT = WORK / "output"
for path in (WORK, REMOTE, OUTPUT):
    path.mkdir(parents=True, exist_ok=True)

if RUN_SCOPE != "AGGREGATE" or CONFIRM_AGGREGATE != "O1S-EXACT-DEV":
    raise RuntimeError("Exact aggregate lock is closed")

## 1 — Refresh repo, GPU compatibility và dependencies

In [ ]:
import gc
import json
import os
import shutil
import subprocess
import sys
import zipfile
from collections import Counter, defaultdict


def run(command: list[str], *, cwd: Path | None = None, env: dict[str, str] | None = None) -> str:
    result = subprocess.run(command, cwd=cwd, env=env, check=False, text=True, capture_output=True)
    if result.returncode:
        raise RuntimeError(f"Command failed: {' '.join(command[:4])}\n{result.stderr[-1800:]}")
    return result.stdout.strip()


from kaggle_secrets import UserSecretsClient

secrets = UserSecretsClient()
try:
    github_token = secrets.get_secret("GITHUB_TOKEN")
    os.environ["MODAL_TOKEN_ID"] = secrets.get_secret("MODAL_TOKEN_ID")
    os.environ["MODAL_TOKEN_SECRET"] = secrets.get_secret("MODAL_TOKEN_SECRET")
except Exception as error:
    raise RuntimeError("Thiếu hoặc chưa bật quyền Kaggle Secrets GitHub/Modal") from error

askpass = WORK / "git-askpass.sh"
askpass.write_text(
    '#!/bin/sh\ncase "$1" in *Username*) printf x-access-token ;; *) printf %s "$GITHUB_TOKEN" ;; esac\n',
    encoding="utf-8",
)
askpass.chmod(0o700)
git_env = os.environ.copy()
git_env.update({"GITHUB_TOKEN": github_token, "GIT_ASKPASS": str(askpass), "GIT_TERMINAL_PROMPT": "0"})
try:
    if (REPO / ".git").is_dir():
        if run(["git", "status", "--porcelain"], cwd=REPO):
            raise RuntimeError("Checkout Kaggle có local changes; Restart Session rồi Run All")
        run(["git", "fetch", "--depth", "1", "origin", REPO_REF], cwd=REPO, env=git_env)
        run(["git", "checkout", "--detach", "FETCH_HEAD"], cwd=REPO)
    else:
        run(["git", "clone", "--depth", "1", "--branch", REPO_REF, REPO_URL, str(REPO)], env=git_env)
finally:
    askpass.unlink(missing_ok=True)


def torch_gpu_probe() -> dict:
    value = run([
        sys.executable,
        "-c",
        "import json,torch; c=torch.cuda.get_device_capability(0); "
        "print(json.dumps({'torch':torch.__version__,'gpu':torch.cuda.get_device_name(0),"
        "'capability':list(c),'arch_list':torch.cuda.get_arch_list()}))",
    ])
    return json.loads(value.splitlines()[-1])


probe_before = torch_gpu_probe()
required_arch = f"sm_{probe_before['capability'][0]}{probe_before['capability'][1]}"
if required_arch not in probe_before["arch_list"]:
    run([
        sys.executable, "-m", "pip", "install", "-q", "--upgrade", "--force-reinstall",
        "torch==2.12.1", "torchvision==0.27.1",
        "--index-url", "https://download.pytorch.org/whl/cu126",
    ])
probe_after = torch_gpu_probe()
if required_arch not in probe_after["arch_list"]:
    raise RuntimeError(f"PyTorch không hỗ trợ GPU được cấp: {probe_after}")

run([sys.executable, "-m", "pip", "install", "-q", "-r", str(REPO / "notebooks/requirements-kaggle-btc-images.txt")])
sys.path.insert(0, str(REPO / "src"))

import numpy as np
import torch
import yaml
from huggingface_hub import snapshot_download
from transformers import AutoModel, AutoTokenizer

from mfe.contracts.ids import build_keyframe_id, parse_keyframe_id
from mfe.core.config import load_config
from mfe.offline.augmentation.io import (
    canonical_hash,
    read_jsonl,
    sha256_file,
    write_json_atomic,
    write_jsonl_atomic,
)
from mfe.offline.augmentation.o1s_aggregation import (
    O1SEventTrace,
    adversarial_counterexample_summary,
    classify_siglip_failure_stage,
    cosine_scores,
    deterministic_best,
    deterministic_injected_rank,
    interval_candidate_indices,
    normalize_feature_rows,
    o1s_math_call_site_ledger,
    validate_o1s_worker_manifest,
)
from mfe.offline.augmentation.oracle_artifacts import OracleArtifactTarget
from mfe.offline.augmentation.representative_oracle import (
    RepresentativeResolvedSample,
    build_representative_scan_plan,
    partition_representative_scan,
    representative_scan_plan_hash,
    validate_scan_completion_marker,
)
from mfe.offline.bundle import BundleLayout, freeze_environment
from mfe.offline.source import load_catalogue
from mfe.offline.transport import ModalVolumeTransport
from mfe.online.translate import build_translator

if not torch.cuda.is_available():
    raise RuntimeError("Notebook 22 cần GPU: bật Accelerator GPU rồi Run All lại")

repo_commit = run(["git", "rev-parse", "HEAD"], cwd=REPO)
producer_config_path = REPO / "config/keyframe_augmentation_kaggle.yaml"
producer_config = yaml.safe_load(producer_config_path.read_text(encoding="utf-8"))
btc = ModalVolumeTransport(producer_config["btc"]["volume"], create_if_missing=False, version=2)
research = ModalVolumeTransport(producer_config["output"]["volume"], create_if_missing=False, version=2)
btc_layout = BundleLayout(producer_config["btc"]["bundle"])
upstream_root = producer_config["output"]["root"]
exact_root = f"{upstream_root}/gate1-o1s-exact-v1"
nb18_root = f"{upstream_root}/gate1-oracle-eval-quang-v1"
lb64_aggregate_root = f"{upstream_root}/gate1-o1s-lb64-aggregate-v1"
output_root = f"{upstream_root}/gate1-o1s-exact-aggregate-v1"
print({"repo_commit": repo_commit, "gpu": torch.cuda.get_device_name(0), "scope": RUN_SCOPE})


def load_npz_vectors(path: Path) -> tuple[list[str], np.ndarray]:
    with np.load(path, allow_pickle=False) as artifact:
        if set(artifact.files) != {"ids", "vectors"}:
            raise RuntimeError(f"Unexpected vector NPZ members: {artifact.files}")
        ids = [str(value) for value in artifact["ids"].tolist()]
        vectors = np.asarray(artifact["vectors"], dtype=np.float32)
    return ids, vectors

## 2 — Khóa lineage và xác minh hai worker manifest exact

In [ ]:
nb18_report_path = REMOTE / "oracle-eval-report.json"
targets_path = REMOTE / "oracle-targets-resolved.jsonl"
nb20_complete_path = REMOTE / "nb20-COMPLETE.json"
research.get(f"{nb18_root}/oracle-eval-report.json", nb18_report_path)
research.get(f"{nb18_root}/oracle-targets-resolved.jsonl", targets_path)
research.get(f"{lb64_aggregate_root}/COMPLETE.json", nb20_complete_path)
if sha256_file(nb18_report_path) != EXPECTED_NB18_REPORT_SHA256:
    raise RuntimeError("Notebook 18 report SHA drift")
if sha256_file(targets_path) != EXPECTED_TARGETS_SHA256:
    raise RuntimeError("Oracle targets SHA drift")
if sha256_file(nb20_complete_path) != EXPECTED_NB20_COMPLETE_SHA256:
    raise RuntimeError("Notebook 20 COMPLETE SHA drift")

nb18 = json.loads(nb18_report_path.read_text(encoding="utf-8"))
nb18_payload = dict(nb18)
nb18_claimed = nb18_payload.pop("report_payload_hash", None)
if nb18_claimed != canonical_hash(nb18_payload):
    raise RuntimeError("Notebook 18 report payload hash mismatch")
if (
    nb18.get("status") != "ORACLE_DIAGNOSTIC_COMPLETE"
    or nb18.get("heldout_accessed") is not False
    or nb18.get("btc_mutated") is not False
):
    raise RuntimeError("Notebook 18 lineage/status invalid")

nb20_complete = json.loads(nb20_complete_path.read_text(encoding="utf-8"))
nb20_payload = dict(nb20_complete)
nb20_claimed = nb20_payload.pop("manifest_payload_hash", None)
if nb20_claimed != canonical_hash(nb20_payload):
    raise RuntimeError("Notebook 20 COMPLETE payload hash mismatch")
if (
    nb20_complete.get("status") != "COMPLETE"
    or nb20_complete.get("heldout_accessed") is not False
    or nb20_complete.get("btc_mutated") is not False
):
    raise RuntimeError("Notebook 20 lineage/status invalid")

targets = [OracleArtifactTarget.model_validate(row) for row in read_jsonl(targets_path)]
if len(targets) != EXPECTED_TARGETS:
    raise RuntimeError(f"Expected {EXPECTED_TARGETS} targets, got {len(targets)}")

queryset_path = REPO / "eval/queryset_dev.jsonl"
if sha256_file(queryset_path) != nb18["input_hashes"]["dev_queryset"]:
    raise RuntimeError("DEV queryset hash differs from notebook 18")
dev_query_ids = {str(row["query_id"]) for row in read_jsonl(queryset_path)}
if len(dev_query_ids) != EXPECTED_DEV_QUERIES:
    raise RuntimeError(f"Expected {EXPECTED_DEV_QUERIES} DEV queries, got {len(dev_query_ids)}")
dev_targets_for_plan = sorted(
    (target for target in targets if target.query_id in dev_query_ids),
    key=lambda target: (target.query_id, target.event_index),
)
if len(dev_targets_for_plan) != EXPECTED_DEV_TARGETS:
    raise RuntimeError("DEV target count drift")
exact_targets = [
    target
    for target in dev_targets_for_plan
    if target.frame_range_end - target.frame_range_start + 1 > MAX_SAMPLES_PER_EVENT
]
if len(exact_targets) != EXPECTED_EXACT_TARGETS:
    raise RuntimeError("Exact target count drift")

by_frame = defaultdict(set)
for target in exact_targets:
    membership = (
        target.query_id,
        target.event_index,
        target.frame_range_start,
        target.frame_range_end,
    )
    for frame_idx in range(target.frame_range_start, target.frame_range_end + 1):
        by_frame[(target.video_id, frame_idx)].add(membership)

exact_plan = []
for (video_id, frame_idx), memberships in sorted(by_frame.items()):
    exact_plan.append({
        "keyframe_id": build_keyframe_id(video_id, frame_idx),
        "video_id": video_id,
        "frame_idx": frame_idx,
        "interval_policy": EXACT_POLICY,
        "memberships": [
            {
                "query_id": query_id,
                "event_index": event_index,
                "frame_range_start": start,
                "frame_range_end": end,
            }
            for query_id, event_index, start, end in sorted(memberships)
        ],
    })
exact_plan_hash = canonical_hash(exact_plan)
if exact_plan_hash != EXPECTED_EXACT_PLAN_HASH or len(exact_plan) != EXPECTED_EXACT_FRAMES:
    raise RuntimeError(f"Exact plan drift: {exact_plan_hash}/{len(exact_plan)}")

catalogue_path = REPO / producer_config["source"]["catalogue"]
catalogue = load_catalogue(catalogue_path)
by_video = defaultdict(list)
for row in exact_plan:
    by_video[row["video_id"]].append(row)
video_costs = {video_id: catalogue[video_id].size for video_id in by_video}
totals = {runner: 0 for runner in RUNNERS}
partitions = {runner: [] for runner in RUNNERS}
for video_id in sorted(by_video, key=lambda value: (-video_costs[value], value)):
    owner = min(RUNNERS, key=lambda value: (totals[value], value))
    partitions[owner].extend(by_video[video_id])
    totals[owner] += video_costs[video_id]
for runner in RUNNERS:
    partitions[runner].sort(key=lambda row: (row["video_id"], row["frame_idx"]))

by_runner_video = {}
for runner, rows in partitions.items():
    grouped = defaultdict(list)
    for row in rows:
        grouped[row["video_id"]].append(row)
    by_runner_video[runner] = dict(grouped)


def validate_exact_worker_manifest(manifest: dict, runner: str) -> None:
    payload = dict(manifest)
    claimed = payload.pop("manifest_payload_hash", None)
    if claimed != canonical_hash(payload):
        raise RuntimeError(f"{runner}: worker manifest payload hash mismatch")
    expected_videos = sorted(by_runner_video[runner])
    expected_frames = sum(len(rows) for rows in by_runner_video[runner].values())
    expected = {
        "status": "COMPLETE",
        "decision_authority": "diagnostic_only",
        "run_id": SCAN_RUN_ID,
        "runner": runner,
        "runners": list(RUNNERS),
        "interval_policy": EXACT_POLICY,
        "full_plan_hash": exact_plan_hash,
        "dev_target_events": EXPECTED_DEV_TARGETS,
        "full_interval_targets": EXPECTED_EXACT_TARGETS,
        "exact_unique_frames": EXPECTED_EXACT_FRAMES,
        "notebook18_report_sha256": EXPECTED_NB18_REPORT_SHA256,
        "targets_sha256": EXPECTED_TARGETS_SHA256,
        "notebook20_complete_sha256": EXPECTED_NB20_COMPLETE_SHA256,
        "assigned_count": len(expected_videos),
        "assigned_frame_count": expected_frames,
        "verified_complete_count": len(expected_videos),
        "repo_commit": EXPECTED_EXACT_REPO_COMMIT,
        "siglip_canary_report_hash": EXPECTED_SIGLIP_CANARY_REPORT_HASH,
        "heldout_accessed": False,
        "btc_mutated": False,
    }
    if any(manifest.get(key) != value for key, value in expected.items()):
        raise RuntimeError(f"{runner}: worker manifest contract mismatch")
    if manifest.get("assigned") != expected_videos:
        raise RuntimeError(f"{runner}: assigned video list mismatch")
    if manifest.get("verified_complete") != expected_videos:
        raise RuntimeError(f"{runner}: verified video list mismatch")
    if manifest.get("failures_this_session"):
        raise RuntimeError(f"{runner}: worker manifest still contains failures")


worker_manifests = {}
for runner in RUNNERS:
    local = REMOTE / "worker-manifests" / f"{runner}.json"
    research.get(f"{exact_root}/worker-manifests/{runner}.json", local)
    manifest = json.loads(local.read_text(encoding="utf-8"))
    validate_exact_worker_manifest(manifest, runner)
    worker_manifests[runner] = manifest

assigned_videos = [video_id for runner in RUNNERS for video_id in worker_manifests[runner]["assigned"]]
if len(assigned_videos) != EXPECTED_EXACT_VIDEOS or len(set(assigned_videos)) != EXPECTED_EXACT_VIDEOS:
    raise RuntimeError("Exact worker assignments are not a disjoint 35-video set")
if sum(item["assigned_frame_count"] for item in worker_manifests.values()) != EXPECTED_EXACT_FRAMES:
    raise RuntimeError("Exact worker frame total is not 32,867")
print({
    "worker_status": {runner: "COMPLETE" for runner in RUNNERS},
    "videos": len(assigned_videos),
    "frames": len(exact_plan),
    "targets": len(exact_targets),
    "plan_hash": exact_plan_hash,
})

## 3 — Verify 35 marker + artifact hash và gộp 32.867 exact vector

In [ ]:
def exact_remote_paths(runner: str, video_id: str) -> dict[str, str]:
    return {
        "images": f"{exact_root}/images/{video_id}.tar",
        "samples": f"{exact_root}/samples/{video_id}.jsonl",
        "siglip": f"{exact_root}/siglip/{video_id}.npz",
        "report": f"{exact_root}/reports/{video_id}.json",
        "marker": f"{exact_root}/markers/{runner}/{video_id}.json",
    }


exact_source_ids = []
exact_source_vectors = []
exact_sample_by_id = {}
exact_source_artifact_hashes = {}
canary_hashes = set()
for runner in RUNNERS:
    for ordinal, video_id in enumerate(worker_manifests[runner]["assigned"], 1):
        paths = exact_remote_paths(runner, video_id)
        marker_local = REMOTE / "markers" / runner / f"{video_id}.json"
        research.get(paths["marker"], marker_local)
        marker = json.loads(marker_local.read_text(encoding="utf-8"))
        marker_payload = dict(marker)
        marker_claimed = marker_payload.pop("marker_payload_hash", None)
        if marker_claimed != canonical_hash(marker_payload):
            raise RuntimeError(f"{runner}/{video_id}: marker payload hash mismatch")
        expected_marker = {
            "status": "COMPLETE",
            "run_id": SCAN_RUN_ID,
            "runner": runner,
            "video_id": video_id,
            "full_plan_hash": exact_plan_hash,
            "interval_policy": EXACT_POLICY,
            "sample_count": len(by_runner_video[runner][video_id]),
            "heldout_accessed": False,
            "btc_mutated": False,
        }
        if any(marker.get(key) != value for key, value in expected_marker.items()):
            raise RuntimeError(f"{runner}/{video_id}: marker contract mismatch")
        expected_artifact_set = {paths[key] for key in ("images", "samples", "siglip", "report")}
        if set(marker.get("artifacts", {})) != expected_artifact_set:
            raise RuntimeError(f"{runner}/{video_id}: unexpected artifact set")
        observed_remote = {path: research.sha256(path) for path in marker["artifacts"]}
        if observed_remote != marker["artifacts"]:
            raise RuntimeError(f"{runner}/{video_id}: remote artifact hash mismatch")
        exact_source_artifact_hashes.update(marker["artifacts"])

        samples_local = REMOTE / "samples" / runner / f"{video_id}.jsonl"
        siglip_local = REMOTE / "siglip" / runner / f"{video_id}.npz"
        report_local = REMOTE / "reports" / runner / f"{video_id}.json"
        for remote_path, local_path in (
            (paths["samples"], samples_local),
            (paths["siglip"], siglip_local),
            (paths["report"], report_local),
        ):
            research.get(remote_path, local_path)
            if sha256_file(local_path) != marker["artifacts"][remote_path]:
                raise RuntimeError(f"{runner}/{video_id}: downloaded artifact hash mismatch")

        samples = read_jsonl(samples_local)
        observed_plan = [
            {
                key: row[key]
                for key in ("keyframe_id", "video_id", "frame_idx", "interval_policy", "memberships")
            }
            for row in samples
        ]
        expected_plan = by_runner_video[runner][video_id]
        if observed_plan != expected_plan:
            raise RuntimeError(f"{runner}/{video_id}: exact sample plan mismatch")
        ids, vectors = load_npz_vectors(siglip_local)
        expected_ids = [row["keyframe_id"] for row in expected_plan]
        if ids != expected_ids or vectors.shape != (len(ids), SIGLIP_DIMENSION):
            raise RuntimeError(f"{runner}/{video_id}: SigLIP identity/shape mismatch")
        if not np.isfinite(vectors).all():
            raise RuntimeError(f"{runner}/{video_id}: non-finite exact vector")

        report = json.loads(report_local.read_text(encoding="utf-8"))
        report_payload = dict(report)
        report_claimed = report_payload.pop("report_payload_hash", None)
        if report_claimed != canonical_hash(report_payload):
            raise RuntimeError(f"{runner}/{video_id}: report payload hash mismatch")
        if (
            report.get("status") != "O1S_EXACT_SIGLIP_VIDEO_COMPLETE"
            or report.get("full_interval_scan") is not True
            or report.get("full_plan_hash") != exact_plan_hash
            or report.get("sample_count") != len(samples)
        ):
            raise RuntimeError(f"{runner}/{video_id}: exact report contract mismatch")
        canary_hashes.add(report["siglip"]["canary_report_hash"])

        for row in samples:
            keyframe_id = str(row["keyframe_id"])
            if keyframe_id in exact_sample_by_id:
                raise RuntimeError(f"Duplicate exact sample: {keyframe_id}")
            exact_sample_by_id[keyframe_id] = row
        exact_source_ids.extend(ids)
        exact_source_vectors.append(vectors)
        if ordinal % 5 == 0 or ordinal == len(worker_manifests[runner]["assigned"]):
            print(f"verified {runner} {ordinal}/{len(worker_manifests[runner]['assigned'])}", flush=True)

if len(exact_source_ids) != EXPECTED_EXACT_FRAMES or len(set(exact_source_ids)) != EXPECTED_EXACT_FRAMES:
    raise RuntimeError("Combined exact source is not 32,867 unique frames")
if canary_hashes != {EXPECTED_SIGLIP_CANARY_REPORT_HASH}:
    raise RuntimeError(f"Exact scan SigLIP canary drift: {canary_hashes}")
exact_source_matrix = normalize_feature_rows(np.concatenate(exact_source_vectors, axis=0))
exact_source_position = {keyframe_id: index for index, keyframe_id in enumerate(exact_source_ids)}
print({"exact_source_rows": len(exact_source_ids), "dimension": exact_source_matrix.shape[1], "remote_integrity": "PASS"})

## 4 — BTC global SigLIP read-only và notebook 20 baseline

In [ ]:
def resolve_base_siglip() -> tuple[Path, dict[str, str]]:
    input_root = Path("/kaggle/input")
    member = "stores/flat/data/vectors/siglip_vectors.npz"
    expanded = sorted(input_root.rglob(member))
    if len(expanded) == 1:
        return expanded[0], {"mode": "expanded_kaggle_dataset", "sha256": sha256_file(expanded[0])}
    if len(expanded) > 1:
        raise RuntimeError(f"Có nhiều hơn một base SigLIP trong Kaggle Input: {expanded}")

    archives = sorted(input_root.rglob("mfe-data-v2.zip"))
    if len(archives) == 1:
        extracted = WORK / "base-input" / member
        with zipfile.ZipFile(archives[0]) as zipped:
            if member not in set(zipped.namelist()):
                raise RuntimeError(f"{archives[0]} thiếu {member}")
            expected_size = zipped.getinfo(member).file_size
            if not extracted.is_file() or extracted.stat().st_size != expected_size:
                extracted.parent.mkdir(parents=True, exist_ok=True)
                partial = extracted.with_suffix(extracted.suffix + ".part")
                partial.unlink(missing_ok=True)
                try:
                    with zipped.open(member) as source, partial.open("wb") as destination:
                        shutil.copyfileobj(source, destination, length=8 * 1024 * 1024)
                    if partial.stat().st_size != expected_size:
                        raise RuntimeError("Extracted base SigLIP size mismatch")
                    partial.replace(extracted)
                finally:
                    partial.unlink(missing_ok=True)
        return extracted, {"mode": "mfe_data_v2_zip", "sha256": sha256_file(extracted)}
    if len(archives) > 1:
        raise RuntimeError(f"Có nhiều hơn một mfe-data-v2.zip trong Kaggle Input: {archives}")

    remote = btc_layout.flat("siglip_vectors")
    if btc.exists(remote):
        downloaded = WORK / "btc-siglip-vectors.npz"
        btc.get(remote, downloaded)
        return downloaded, {"mode": "keyframes_btc_flat", "sha256": sha256_file(downloaded)}
    raise RuntimeError(
        "Thiếu BTC global SigLIP. Hãy Add Input đúng Dataset mfe-data-v2 đã dùng ở "
        "notebook 18 (có stores/flat/data/vectors/siglip_vectors.npz hoặc mfe-data-v2.zip), "
        "rồi Save Version lại. Không cần chạy lại notebook 19."
    )


base_siglip_path, base_siglip_source = resolve_base_siglip()
expected_base_siglip_hash = nb18["input_hashes"]["base_siglip"]
if sha256_file(base_siglip_path) != expected_base_siglip_hash:
    raise RuntimeError("BTC global SigLIP hash differs from notebook 18 baseline")
btc_ids_raw, btc_vectors_raw = load_npz_vectors(base_siglip_path)
btc_ids = [str(value) for value in btc_ids_raw]
btc_matrix = normalize_feature_rows(np.asarray(btc_vectors_raw, dtype=np.float32))
if len(btc_ids) != EXPECTED_BTC_SIGLIP_ROWS or len(set(btc_ids)) != EXPECTED_BTC_SIGLIP_ROWS:
    raise RuntimeError("BTC global SigLIP identity count drift")
if btc_matrix.shape != (EXPECTED_BTC_SIGLIP_ROWS, SIGLIP_DIMENSION):
    raise RuntimeError(f"BTC global SigLIP shape drift: {btc_matrix.shape}")

queryset_path = REPO / "eval/queryset_dev.jsonl"
if sha256_file(queryset_path) != nb18["input_hashes"]["dev_queryset"]:
    raise RuntimeError("DEV queryset hash differs from notebook 18")
queries = read_jsonl(queryset_path)
query_by_id = {str(row["query_id"]): row for row in queries}
if len(query_by_id) != EXPECTED_DEV_QUERIES:
    raise RuntimeError(f"Expected {EXPECTED_DEV_QUERIES} DEV queries, got {len(query_by_id)}")
dev_targets = sorted(
    (target for target in targets if target.query_id in query_by_id),
    key=lambda target: (target.query_id, target.event_index),
)
if len(dev_targets) != EXPECTED_DEV_TARGETS:
    raise RuntimeError(f"Expected {EXPECTED_DEV_TARGETS} DEV target events, got {len(dev_targets)}")
print({"btc_rows": len(btc_ids), "dev_queries": len(query_by_id), "dev_target_events": len(dev_targets), "btc_mode": "READ_ONLY", "base_siglip_source": base_siglip_source})


lb64_events_path = REMOTE / "lb64-o1s-event-results.jsonl"
lb64_report_path = REMOTE / "lb64-o1s-aggregate-report.json"
research.get(f"{lb64_aggregate_root}/o1s-event-results.jsonl", lb64_events_path)
research.get(f"{lb64_aggregate_root}/o1s-aggregate-report.json", lb64_report_path)
for local in (lb64_events_path, lb64_report_path):
    expected_hash = nb20_complete["output_hashes"].get(local.name.removeprefix("lb64-"))
    if expected_hash is None or sha256_file(local) != expected_hash:
        raise RuntimeError(f"Notebook 20 artifact hash mismatch: {local.name}")
lb64_report = json.loads(lb64_report_path.read_text(encoding="utf-8"))
if lb64_report.get("status") != "O1S_LB64_AGGREGATE_COMPLETE":
    raise RuntimeError("Notebook 20 aggregate report status invalid")
lb64_events = [O1SEventTrace.model_validate(row) for row in read_jsonl(lb64_events_path)]
lb64_by_event = {(row.query_id, row.event_index): row for row in lb64_events}
if len(lb64_events) != EXPECTED_DEV_TARGETS or len(lb64_by_event) != EXPECTED_DEV_TARGETS:
    raise RuntimeError("Notebook 20 event result count drift")

## 5 — Query text production contract và SigLIP text tower

In [ ]:
def event_query_text(query: dict, event_index: int) -> str:
    if query["type"] == "TRAKE":
        texts = query.get("event_texts") or []
        if not 0 <= event_index < len(texts):
            raise RuntimeError(f"{query['query_id']}: missing TRAKE event text {event_index}")
        return str(texts[event_index])
    if query["type"] == "QA":
        return str(query.get("retrieval_text") or query["text"])
    return str(query["text"])


translation_root = WORK / "translation-root"
translation_cache_input = REPO / "eval/translations.json"
translation_cache_runtime = translation_root / "eval/translations.json"
translation_cache_runtime.parent.mkdir(parents=True, exist_ok=True)
shutil.copy2(translation_cache_input, translation_cache_runtime)
translation_cache_input_hash = sha256_file(translation_cache_input)
config = load_config(REPO / "config/btc_b1.yaml", repo_root=REPO)
translator = build_translator(config, repo_root=translation_root)
if translator is None:
    raise RuntimeError("Production translation contract is disabled")

translation_rows = []
event_text_en = {}
for target in dev_targets:
    key = (target.query_id, target.event_index)
    query = query_by_id[target.query_id]
    text_vi = event_query_text(query, target.event_index)
    translation = translator.translate(text_vi, manual=query.get("text_en"))
    if translation is None:
        raise RuntimeError(f"{target.query_id}/{target.event_index}: no English SigLIP query")
    event_text_en[key] = translation.text_en
    translation_rows.append({
        "query_id": target.query_id,
        "event_index": target.event_index,
        "text_vi": text_vi,
        "text_en": translation.text_en,
        "tier": translation.tier,
        "model": translation.model,
    })
translator.save_cache()
translation_cache_runtime_hash = sha256_file(translation_cache_runtime)
translations_path = write_jsonl_atomic(OUTPUT / "translations-used.jsonl", translation_rows)
translation_usage = dict(translator.usage)
del translator
gc.collect()

snapshot_dir = Path(snapshot_download(CHECKPOINT, revision=MODEL_REVISION))
model_file_hashes = {
    str(path.relative_to(snapshot_dir)): sha256_file(path)
    for path in sorted(snapshot_dir.rglob("*"))
    if path.is_file()
}
model_snapshot_hash = canonical_hash(model_file_hashes)
tokenizer = AutoTokenizer.from_pretrained(snapshot_dir, local_files_only=True)
model = AutoModel.from_pretrained(snapshot_dir, local_files_only=True).eval().cuda()

ordered_event_keys = [(target.query_id, target.event_index) for target in dev_targets]
query_vectors = []
for start in range(0, len(ordered_event_keys), TEXT_BATCH_SIZE):
    keys = ordered_event_keys[start : start + TEXT_BATCH_SIZE]
    batch = tokenizer(
        [event_text_en[key] for key in keys],
        padding="max_length",
        max_length=TEXT_MAX_LENGTH,
        truncation=True,
        return_tensors="pt",
    ).to("cuda")
    with torch.inference_mode():
        output = model.get_text_features(**batch)
        pooled = getattr(output, "pooler_output", None)
        features = pooled if pooled is not None else output
    query_vectors.append(features.float().cpu().numpy())
query_matrix = normalize_feature_rows(np.concatenate(query_vectors, axis=0))
if query_matrix.shape != (EXPECTED_DEV_TARGETS, SIGLIP_DIMENSION):
    raise RuntimeError(f"SigLIP query matrix shape drift: {query_matrix.shape}")
del model, tokenizer, query_vectors, batch, output, pooled, features
gc.collect()
torch.cuda.empty_cache()
print({"translated_events": len(translation_rows), "translation_usage": translation_usage, "model_snapshot_hash": model_snapshot_hash})

## 6 — Exact best source, BTC baseline, rank và LB64 comparison

In [ ]:
# Một GPU matmul cho 71 query-event × 176.707 BTC vectors.
with torch.inference_mode():
    btc_tensor = torch.from_numpy(btc_matrix).to("cuda")
    query_tensor = torch.from_numpy(query_matrix).to("cuda")
    btc_score_matrix = (query_tensor @ btc_tensor.T).float().cpu().numpy()
del btc_tensor, query_tensor
torch.cuda.empty_cache()

exact_membership_positions = defaultdict(list)
for keyframe_id, sample in exact_sample_by_id.items():
    position = exact_source_position[keyframe_id]
    for membership in sample["memberships"]:
        exact_membership_positions[(membership["query_id"], membership["event_index"])].append(position)

b0_scores = dict(zip(nb18["variants"]["B0"]["query_ids"], nb18["variants"]["B0"]["scores"], strict=True))
o1r_scores = dict(zip(nb18["variants"]["O1-R-FINAL"]["query_ids"], nb18["variants"]["O1-R-FINAL"]["scores"], strict=True))
exact_event_keys = {(target.query_id, target.event_index) for target in exact_targets}

event_results = []
stage_traces = []
best_source_candidates = []
exact_vs_lb64 = []
for event_position, target in enumerate(dev_targets):
    event_key = (target.query_id, target.event_index)
    lb64 = lb64_by_event[event_key]
    query = query_by_id[target.query_id]
    start_frame, end_frame = target.frame_range_start, target.frame_range_end
    global_scores = btc_score_matrix[event_position]
    query_vector = query_matrix[event_position]

    interval_positions = interval_candidate_indices(
        btc_ids,
        video_id=target.video_id,
        start_frame=start_frame,
        end_frame=end_frame,
    )
    if len(interval_positions):
        interval_ids = [btc_ids[index] for index in interval_positions]
        interval_scores = global_scores[interval_positions]
        best_btc_id, best_btc_score, _ = deterministic_best(interval_ids, interval_scores)
        best_btc_frame = parse_keyframe_id(best_btc_id)[1]
        best_btc_rank = deterministic_injected_rank(
            btc_ids,
            global_scores,
            candidate_id=best_btc_id,
            candidate_score=best_btc_score,
        )
    else:
        best_btc_id = None
        best_btc_score = None
        best_btc_frame = None
        best_btc_rank = None

    if best_btc_frame != lb64.best_btc_frame_idx or best_btc_rank != lb64.best_btc_rank:
        raise RuntimeError(f"{event_key}: BTC baseline identity/rank drift from notebook 20")
    old_btc_score = lb64.best_btc_score_by_stage["siglip_dense"]
    if best_btc_score is None:
        if old_btc_score is not None:
            raise RuntimeError(f"{event_key}: BTC temporal state drift")
    elif old_btc_score is None or not np.isclose(best_btc_score, old_btc_score, rtol=0.0, atol=1e-5):
        raise RuntimeError(f"{event_key}: BTC score drift from notebook 20")

    global_order = np.lexsort((np.asarray(btc_ids), -global_scores))
    interval_set = set(interval_positions.tolist())
    outside_position = next(index for index in global_order if int(index) not in interval_set)
    top_outside_id = btc_ids[int(outside_position)]
    top_outside_score = float(global_scores[int(outside_position)])
    top_outside_rank = deterministic_injected_rank(
        btc_ids,
        global_scores,
        candidate_id=top_outside_id,
        candidate_score=top_outside_score,
    )

    if event_key in exact_event_keys:
        source_positions = sorted(set(exact_membership_positions[event_key]))
        expected_count = end_frame - start_frame + 1
        if len(source_positions) != expected_count:
            raise RuntimeError(f"{event_key}: exact membership count {len(source_positions)} != {expected_count}")
        local_source_ids = [exact_source_ids[index] for index in source_positions]
        local_source_scores = cosine_scores(exact_source_matrix[source_positions], query_vector)
        best_source_id, best_source_score, _ = deterministic_best(local_source_ids, local_source_scores)
        best_source_frame = parse_keyframe_id(best_source_id)[1]
        best_source_rank = deterministic_injected_rank(
            btc_ids,
            global_scores,
            candidate_id=best_source_id,
            candidate_score=best_source_score,
        )
        sampling_policy = EXACT_POLICY
        source_provenance_hash = canonical_hash(exact_source_artifact_hashes)
        lb64_source_score = float(lb64.best_source_score_by_stage["siglip_dense"])
        exact_vs_lb64.append({
            "query_id": target.query_id,
            "event_index": target.event_index,
            "video_id": target.video_id,
            "interval_length": expected_count,
            "lb64_best_source_frame_idx": lb64.best_source_frame_idx,
            "exact_best_source_frame_idx": best_source_frame,
            "lb64_best_source_score": lb64_source_score,
            "exact_best_source_score": best_source_score,
            "exact_minus_lb64_source_score": best_source_score - lb64_source_score,
            "lb64_best_source_rank": lb64.best_source_rank,
            "exact_best_source_rank": best_source_rank,
            "rank_gain_positive_is_better": lb64.best_source_rank - best_source_rank,
            "best_frame_changed": best_source_frame != lb64.best_source_frame_idx,
        })
    else:
        best_source_frame = lb64.best_source_frame_idx
        best_source_id = build_keyframe_id(target.video_id, best_source_frame)
        best_source_score = float(lb64.best_source_score_by_stage["siglip_dense"])
        best_source_rank = lb64.best_source_rank
        sampling_policy = LB64_POLICY
        source_provenance_hash = EXPECTED_NB20_COMPLETE_SHA256

    o1s_delta = None if best_btc_score is None else best_source_score - best_btc_score
    failure_stage = classify_siglip_failure_stage(
        has_btc_in_interval=best_btc_id is not None,
        source_signal_delta=o1s_delta,
        best_source_rank=best_source_rank,
        best_btc_rank=best_btc_rank,
        exposure_depth=EXPOSURE_DEPTH,
        improvement_epsilon=SIGNAL_IMPROVEMENT_EPSILON,
    )
    source_stage_scores = {"siglip_dense": best_source_score, "fusion": None, "reranker": None, "scorer": None}
    btc_stage_scores = {"siglip_dense": best_btc_score, "fusion": None, "reranker": None, "scorer": None}
    provenance = {
        "exact_plan": exact_plan_hash,
        "notebook18_report": EXPECTED_NB18_REPORT_SHA256,
        "notebook20_complete": EXPECTED_NB20_COMPLETE_SHA256,
        "oracle_targets": EXPECTED_TARGETS_SHA256,
        "dev_queryset": nb18["input_hashes"]["dev_queryset"],
        "btc_siglip": expected_base_siglip_hash,
        "source_artifacts": source_provenance_hash,
        "model_snapshot": model_snapshot_hash,
        "translation_cache_runtime": translation_cache_runtime_hash,
    }
    result = O1SEventTrace(
        query_id=target.query_id,
        query_type=query["type"],
        event_index=target.event_index,
        video_id=target.video_id,
        gt_interval_inclusive=(start_frame, end_frame),
        sampling_policy=sampling_policy,
        full_interval_scan=True,
        best_btc_frame_idx=best_btc_frame,
        best_btc_rank=best_btc_rank,
        best_btc_score_by_stage=btc_stage_scores,
        top_btc_outside_interval_id=top_outside_id,
        top_btc_outside_interval_rank=top_outside_rank,
        top_btc_outside_interval_score=top_outside_score,
        best_source_frame_idx=best_source_frame,
        best_source_rank=best_source_rank,
        best_source_score_by_stage=source_stage_scores,
        o1_t_delta=None,
        o1_s_delta=o1s_delta,
        o1_r_delta=float(o1r_scores[target.query_id] - b0_scores[target.query_id]),
        failure_stage=failure_stage,
        failure_stage_status="MEASURED_FULL_INTERVAL_SIGLIP_ONLY_DOWNSTREAM_BLOCKED",
        provenance_hashes=provenance,
    )
    event_results.append(result)
    best_source_candidates.append({
        "query_id": target.query_id,
        "event_index": target.event_index,
        "query_type": query["type"],
        "video_id": target.video_id,
        "keyframe_id": best_source_id,
        "frame_idx": best_source_frame,
        "siglip_score": best_source_score,
        "injected_rank": best_source_rank,
        "o1_s_delta": o1s_delta,
        "sampling_policy": sampling_policy,
        "full_interval_scan": True,
    })
    for stage, status in (
        ("siglip_dense", "MEASURED_FULL_INTERVAL"),
        ("fusion", "BLOCKED_MISSING_FEATURE_PARITY"),
        ("reranker", "BLOCKED_NO_EXACT_STAGE_TRACE"),
        ("scorer", "BLOCKED_NO_OFFICIAL_AB"),
    ):
        stage_traces.append({
            "query_id": target.query_id,
            "event_index": target.event_index,
            "stage": stage,
            "status": status,
            "best_source_score": source_stage_scores[stage],
            "best_btc_score": btc_stage_scores[stage],
            "best_source_rank": best_source_rank if stage == "siglip_dense" else None,
            "best_btc_rank": best_btc_rank if stage == "siglip_dense" else None,
        })
    if (event_position + 1) % 10 == 0 or event_position + 1 == len(dev_targets):
        print(f"measured exact aggregate {event_position + 1}/{len(dev_targets)}", flush=True)

events_path = write_jsonl_atomic(OUTPUT / "o1s-exact-event-results.jsonl", event_results)
traces_path = write_jsonl_atomic(OUTPUT / "o1s-exact-stage-traces.jsonl", stage_traces)
candidates_path = write_jsonl_atomic(OUTPUT / "o1s-exact-best-source-candidates.jsonl", best_source_candidates)
comparison_path = write_jsonl_atomic(OUTPUT / "o1s-exact-vs-lb64.jsonl", exact_vs_lb64)
if len(event_results) != EXPECTED_DEV_TARGETS or len(exact_vs_lb64) != EXPECTED_EXACT_TARGETS:
    raise RuntimeError("Exact aggregate output count drift")
print({"events": len(event_results), "exact_comparisons": len(exact_vs_lb64), "stage_traces": len(stage_traces)})

## 7 — Aggregate report, headroom decision và provenance

In [ ]:
deltas = [row.o1_s_delta for row in event_results if row.o1_s_delta is not None]
source_better = [value for value in deltas if value > SIGNAL_IMPROVEMENT_EPSILON]
rank_improved = [
    row for row in event_results
    if row.best_btc_rank is not None and row.best_source_rank < row.best_btc_rank
]
exact_beats_lb64 = [
    row for row in exact_vs_lb64
    if row["exact_minus_lb64_source_score"] > SIGNAL_IMPROVEMENT_EPSILON
]
adversarial_summary = adversarial_counterexample_summary(
    event_results,
    exposure_depth=EXPOSURE_DEPTH,
    improvement_epsilon=SIGNAL_IMPROVEMENT_EPSILON,
)
math_path = write_json_atomic(
    OUTPUT / "math-call-sites.json",
    {"schema_version": 1, "call_sites": o1s_math_call_site_ledger()},
)

dirty_after_clone = run(["git", "status", "--porcelain"], cwd=REPO).splitlines()
if dirty_after_clone:
    raise RuntimeError(f"Notebook 22 checkout became dirty: {dirty_after_clone[:10]}")
workspace_safety = {
    "schema_version": 1,
    "status": "PASS",
    "repo_ref": REPO_REF,
    "repo_commit": repo_commit,
    "repo_clean": True,
    "scope": "DEV_ONLY_O1S_EXACT_AGGREGATE",
    "read_only_volumes": [producer_config["btc"]["volume"]],
    "research_volume_read_prefixes": [nb18_root, lb64_aggregate_root, exact_root],
    "research_volume_write_prefix": output_root,
    "heldout_accessed": False,
    "btc_mutated": False,
    "live_store_mutated": False,
}
workspace_safety_path = write_json_atomic(OUTPUT / "workspace-safety-manifest.json", workspace_safety)

input_ledger = {
    "schema_version": 1,
    "command": "Kaggle Save Version / Run All: notebook 22 exact O1-S aggregate",
    "split": "dev",
    "repo_commit": repo_commit,
    "input_hashes": {
        "notebook18_report": EXPECTED_NB18_REPORT_SHA256,
        "notebook20_complete": EXPECTED_NB20_COMPLETE_SHA256,
        "oracle_targets": EXPECTED_TARGETS_SHA256,
        "exact_plan": exact_plan_hash,
        "worker_manifests": canonical_hash({runner: worker_manifests[runner]["manifest_payload_hash"] for runner in RUNNERS}),
        "exact_source_artifacts": canonical_hash(exact_source_artifact_hashes),
        "btc_siglip": expected_base_siglip_hash,
        "dev_queryset": nb18["input_hashes"]["dev_queryset"],
        "translation_cache_input": translation_cache_input_hash,
        "translation_cache_runtime": translation_cache_runtime_hash,
        "model_snapshot": model_snapshot_hash,
        "workspace_safety_manifest": sha256_file(workspace_safety_path),
    },
    "input_sources": {"btc_siglip": base_siglip_source},
    "config": {
        "exact_policy": EXACT_POLICY,
        "short_interval_policy": LB64_POLICY,
        "exposure_depth": EXPOSURE_DEPTH,
        "signal_improvement_epsilon": SIGNAL_IMPROVEMENT_EPSILON,
        "text_max_length": TEXT_MAX_LENGTH,
        "checkpoint": CHECKPOINT,
        "revision": MODEL_REVISION,
    },
    "heldout_accessed": False,
    "btc_access": "READ_ONLY",
    "btc_mutated": False,
}
input_ledger_path = write_json_atomic(OUTPUT / "input-ledger.json", input_ledger)

headroom_status = (
    "HEADROOM_OBSERVED_AT_FULL_INTERVAL_SIGLIP_DENSE"
    if source_better
    else "NO_SIGLIP_HEADROOM_OBSERVED_DOWNSTREAM_NOT_EXCLUDED"
)
aggregate_report = {
    "schema_version": 1,
    "status": "O1S_EXACT_AGGREGATE_COMPLETE",
    "decision_authority": "diagnostic_only",
    "split": "dev",
    "run_id": RUN_ID,
    "scan_run_id": SCAN_RUN_ID,
    "events": len(event_results),
    "queries": len({row.query_id for row in event_results}),
    "exact_long_interval_events": EXPECTED_EXACT_TARGETS,
    "inherited_full_short_interval_events": EXPECTED_DEV_TARGETS - EXPECTED_EXACT_TARGETS,
    "exact_scan_videos": EXPECTED_EXACT_VIDEOS,
    "exact_scan_frames": EXPECTED_EXACT_FRAMES,
    "full_interval_events": sum(row.full_interval_scan for row in event_results),
    "events_without_interval_btc": sum(row.best_btc_frame_idx is None for row in event_results),
    "events_with_source_signal_gain": len(source_better),
    "events_with_rank_improvement": len(rank_improved),
    "exact_events_beating_lb64": len(exact_beats_lb64),
    "exact_events_changing_best_frame": sum(row["best_frame_changed"] for row in exact_vs_lb64),
    "mean_exact_minus_lb64_source_score": float(np.mean([row["exact_minus_lb64_source_score"] for row in exact_vs_lb64])),
    "mean_o1_s_siglip_delta_represented_events": float(np.mean(deltas)) if deltas else None,
    "median_o1_s_siglip_delta_represented_events": float(np.median(deltas)) if deltas else None,
    "max_o1_s_siglip_delta": float(max(deltas)) if deltas else None,
    "failure_stage_counts": dict(sorted(Counter(row.failure_stage for row in event_results).items())),
    "adversarial_counterexamples": adversarial_summary,
    "headroom_status": headroom_status,
    "oracle_status": {
        "O1-T": "MEASURED_BY_NOTEBOOK_11",
        "O1-S": "MEASURED_FULL_INTERVAL_SIGLIP_DENSE_ONLY",
        "O1-R": "MEASURED_FINAL_CANDIDATE_FORCE_RANK_BY_NOTEBOOK_18",
    },
    "full_o1s_status": "MEASURED_FULL_INTERVAL_SIGLIP_DENSE_ONLY_DOWNSTREAM_BLOCKED",
    "feature_parity_status": "BLOCKED_MISSING_OCR_CAPTION_BGE_AND_DOWNSTREAM_FINGERPRINTS",
    "official_score_status": "UNVERIFIED_NO_EQUAL_BUDGET_OFFICIAL_DEV_AB",
    "notebook23_status": "BLOCKED_PENDING_FULL_FEATURE_PARITY",
    "translation_usage": translation_usage,
    "model": {
        "checkpoint": CHECKPOINT,
        "revision": MODEL_REVISION,
        "snapshot_hash": model_snapshot_hash,
        "text_padding": "max_length_64",
    },
    "environment": freeze_environment(),
    "input_ledger_sha256": sha256_file(input_ledger_path),
    "math_ledger_sha256": sha256_file(math_path),
    "prohibited_inference": "Exact SigLIP headroom alone cannot establish official-score uplift or authorize production.",
    "next": "Review headroom slices, then complete OCR/caption/BGE/downstream parity before notebook 23.",
    "heldout_accessed": False,
    "btc_mutated": False,
}
aggregate_report["report_payload_hash"] = canonical_hash(aggregate_report)
report_path = write_json_atomic(OUTPUT / "o1s-exact-aggregate-report.json", aggregate_report)

output_hashes = {
    path.name: sha256_file(path)
    for path in (
        events_path,
        traces_path,
        candidates_path,
        comparison_path,
        translations_path,
        math_path,
        workspace_safety_path,
        input_ledger_path,
        report_path,
    )
}
measurement_ledger = {
    "schema_version": 1,
    "evidence_label": "MEASURED",
    "command": input_ledger["command"],
    "input_hashes": input_ledger["input_hashes"],
    "config_hash": canonical_hash(input_ledger["config"]),
    "output_hashes": output_hashes,
    "heldout_accessed": False,
    "btc_mutated": False,
}
measurement_path = write_json_atomic(OUTPUT / "measurement-ledger.json", measurement_ledger)
output_hashes[measurement_path.name] = sha256_file(measurement_path)
complete = {
    "schema_version": 1,
    "status": "COMPLETE",
    "run_id": RUN_ID,
    "report_payload_hash": aggregate_report["report_payload_hash"],
    "output_hashes": output_hashes,
    "heldout_accessed": False,
    "btc_mutated": False,
}
complete["manifest_payload_hash"] = canonical_hash(complete)
complete_path = write_json_atomic(OUTPUT / "COMPLETE.json", complete)
print(json.dumps({
    "status": aggregate_report["status"],
    "events": aggregate_report["events"],
    "exact_events": aggregate_report["exact_long_interval_events"],
    "source_signal_gain": aggregate_report["events_with_source_signal_gain"],
    "rank_improvement": aggregate_report["events_with_rank_improvement"],
    "exact_beats_lb64": aggregate_report["exact_events_beating_lb64"],
    "headroom_status": aggregate_report["headroom_status"],
    "notebook23_status": aggregate_report["notebook23_status"],
}, ensure_ascii=False, indent=2))

## 8 — Publish research-only outputs rồi verify remote

In [ ]:
uploads = [
    (events_path, f"{output_root}/o1s-exact-event-results.jsonl"),
    (traces_path, f"{output_root}/o1s-exact-stage-traces.jsonl"),
    (candidates_path, f"{output_root}/o1s-exact-best-source-candidates.jsonl"),
    (comparison_path, f"{output_root}/o1s-exact-vs-lb64.jsonl"),
    (translations_path, f"{output_root}/translations-used.jsonl"),
    (math_path, f"{output_root}/math-call-sites.json"),
    (workspace_safety_path, f"{output_root}/workspace-safety-manifest.json"),
    (input_ledger_path, f"{output_root}/input-ledger.json"),
    (report_path, f"{output_root}/o1s-exact-aggregate-report.json"),
    (measurement_path, f"{output_root}/measurement-ledger.json"),
]
research.put_many(uploads)
for local, remote_path in uploads:
    if research.sha256(remote_path) != sha256_file(local):
        raise RuntimeError(f"Remote output hash mismatch: {remote_path}")

complete_remote = f"{output_root}/COMPLETE.json"
research.put(complete_path, complete_remote)
if research.sha256(complete_remote) != sha256_file(complete_path):
    raise RuntimeError("Remote COMPLETE marker hash mismatch")
print(json.dumps({
    "status": "O1S_EXACT_AGGREGATE_COMPLETE",
    "events": len(event_results),
    "exact_events": len(exact_vs_lb64),
    "remote_output": output_root,
    "manifest_sha256": sha256_file(complete_path),
    "next": aggregate_report["next"],
    "btc_mutated": False,
    "heldout_accessed": False,
}, ensure_ascii=False, indent=2))

## Cách đọc kết quả

- `headroom_status=HEADROOM_OBSERVED...`: exact source có tín hiệu tốt hơn BTC ở
  SigLIP; vẫn chưa chứng minh điểm official tăng.
- `exact_events_beating_lb64`: số interval mà quét đủ tìm được source tốt hơn
  sampling 64 frame; đây là mức LB64 đã bỏ sót.
- `notebook23_status` vẫn `BLOCKED` cho đến khi OCR/caption/BGE và downstream
  fingerprint parity hoàn tất.
- Thành công kỹ thuật chỉ khi log cuối có `O1S_EXACT_AGGREGATE_COMPLETE` và
  remote `COMPLETE.json` đã verify.